# 03 — LRP for Visual Modality

Computes LRP spatial heatmaps over **image pixels** for `image` and `image+text` modes.

Uses zennit Gamma rule on `pixel_values` (via `Qwen3-VL` visual encoder's Conv3d layers).

**No generation.** Loads cached outputs from `01_inference.ipynb`.

Outputs:
- 2D numpy heatmap (float32) per sample → `data/lrp_results/visual_<mode>/<id>.npy`
- Side-by-side overlay PNG → `data/lrp_results/visual_<mode>/overlay/<id>.png`
- JSON metadata → `data/lrp_results/visual_<mode>/<id>.json`

In [ ]:
CONFIG = {
    "dataset_path": "../../datasets/yt/",
    "language": "uk",
    "model_id": "Qwen/Qwen3-VL-4B-Instruct",
    "inference_cache_dir": "./data/inference_cache",
    "output_dir": "./data/lrp_results",
    "modes": ["image", "image+text"],
    "image_size": (256, 256),
    "conv_gamma": 0.1,
    "lin_gamma": 0.1,
    "lrp_sigma": 3,
    "lrp_outlier_percentile": 99,
    "seed": 42,
}

In [2]:
import json
import traceback
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import zennit.rules as z_rules
from PIL import Image
from scipy.ndimage import gaussian_filter
from torch.nn import Dropout, LayerNorm
from tqdm import tqdm
from transformers import AutoProcessor
from transformers.models.qwen3_vl import modeling_qwen3_vl
from transformers.models.qwen3_vl.modeling_qwen3_vl import Qwen3VLTextMLP, Qwen3VLTextRMSNorm
from zennit.composites import LayerMapComposite

from lxt.efficient import monkey_patch
from lxt.efficient.patches import (
    dropout_forward, gated_mlp_forward, layer_norm_forward,
    patch_attention, patch_method, rms_norm_forward,
)
from lxt.efficient.zennit_patches import monkey_patch_zennit

OUT_DIR = Path(CONFIG["output_dir"])
CACHE_DIR = Path(CONFIG["inference_cache_dir"])
IMG_DIR = Path(CONFIG["dataset_path"]) / "images"

Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


In [3]:
attn_lrp = {
    Qwen3VLTextMLP: partial(patch_method, gated_mlp_forward),
    Qwen3VLTextRMSNorm: partial(patch_method, rms_norm_forward),
    Dropout: partial(patch_method, dropout_forward),
    modeling_qwen3_vl: patch_attention,
    LayerNorm: partial(patch_method, layer_norm_forward),
}
monkey_patch(modeling_qwen3_vl, patch_map=attn_lrp, verbose=True)
monkey_patch_zennit(verbose=True)

Patched Qwen3VLTextMLP
Patched Qwen3VLTextRMSNorm
Patched Dropout
Patching attention function: flash_attention_3
Patching attention function: flash_attention_2
Patching attention function: flex_attention
Patching attention function: paged_attention
Patching attention function: sdpa
Patching attention function: sdpa_paged
Patching attention function: eager_paged
Patched transformers.models.qwen3_vl.modeling_qwen3_vl
Patched LayerNorm
Patched Zennit BasicHook's forward
Patched Zennit BasicHook's backward


In [4]:
model = modeling_qwen3_vl.Qwen3VLForConditionalGeneration.from_pretrained(
    CONFIG["model_id"], device_map="cuda", dtype=torch.bfloat16,
)
model.eval()
processor = AutoProcessor.from_pretrained(CONFIG["model_id"])
print(f"Model loaded: {CONFIG['model_id']}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen3-VL-4B-Instruct


In [5]:
def find_label_positions(seq, target_labels, input_len):
    positions = []
    for label in target_labels:
        label_ids = processor.tokenizer.encode(label, add_special_tokens=False)
        for idx in range(input_len, len(seq) - len(label_ids) + 1):
            if seq[idx:idx+len(label_ids)] == label_ids:
                positions.extend(range(idx, idx+len(label_ids)))
                break
    return sorted(set(positions))


def compute_relevance_map(pixel_values_grad, pixel_values, grid_t, grid_h, grid_w):
    """Reshape pixel_values gradient×input back to spatial heatmap."""
    sigma = CONFIG["lrp_sigma"]
    outlier_pct = CONFIG["lrp_outlier_percentile"]
    patch_size = 16
    temporal_patch_size = 2
    merge_size = 2
    channels = 3

    rel = (pixel_values * pixel_values_grad).detach().cpu().float()
    rel = rel.reshape(
        grid_t, grid_h // merge_size, grid_w // merge_size,
        merge_size, merge_size, channels,
        temporal_patch_size, patch_size, patch_size,
    )
    rel = rel.permute(0, 6, 5, 1, 3, 7, 2, 4, 8)
    img_h = grid_h * patch_size
    img_w = grid_w * patch_size
    rel = rel.reshape(grid_t * temporal_patch_size, channels, img_h, img_w)
    rel = rel.sum(dim=(0, 1)).numpy()

    if outlier_pct < 100:
        vmax = np.percentile(np.abs(rel), outlier_pct)
        rel = np.clip(rel, -vmax, vmax)
    if sigma > 0:
        rel = gaussian_filter(rel, sigma=sigma)
    rel = rel / (np.abs(rel).max() + 1e-8)
    return rel


def save_overlay(image, heatmap, save_path, alpha=0.6):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(image); axes[0].set_title("Original"); axes[0].axis("off")
    axes[1].imshow(image)
    axes[1].imshow(heatmap, cmap="seismic", alpha=alpha, vmin=-1, vmax=1)
    axes[1].set_title("LRP Heatmap (red=relevant)"); axes[1].axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


def compute_visual_lrp(record, mode):
    sample_id = record["id"]
    pred_labels = record.get("pred_labels", [])
    if not pred_labels:
        return None

    image_filename = record.get("image") or record.get("id", "") + ".png"
    # The record from 01 has field 'image' only if we stored it — we reconstruct from sample id
    # We stored input_ids; re-encode to get pixel_values fresh
    output_ids = torch.tensor(record["output_ids"], dtype=torch.long).unsqueeze(0).to("cuda")
    input_len = record["input_len"]
    seq = record["output_ids"]
    grid_thw = record.get("image_grid_thw")
    if grid_thw is None:
        return None
    grid_t, grid_h, grid_w = int(grid_thw[0]), int(grid_thw[1]), int(grid_thw[2])

    label_positions = find_label_positions(seq, pred_labels, input_len)
    if not label_positions:
        return None

    logit_positions = [p - 1 for p in label_positions if p > 0]
    target_token_ids = [seq[p] for p in label_positions if p > 0]

    # Re-encode image to get pixel_values tensor
    # The image filename is stored in the dataset; we load by sample id
    img_path = record.get("_image_path")  # populated below if needed
    if img_path is None:
        return None

    image = Image.open(img_path).convert("RGB").resize(CONFIG["image_size"])
    # Minimal re-encode just for pixel_values
    dummy_text = processor.apply_chat_template(
        [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "x"}]}],
        tokenize=False, add_generation_prompt=True,
    )
    tmp_inputs = processor(
        text=[dummy_text], images=[image], padding=True, return_tensors="pt"
    ).to("cuda")

    torch.cuda.empty_cache()
    pixel_values = tmp_inputs.pixel_values.float().clone().detach().requires_grad_(True)

    zennit_comp = LayerMapComposite([
        (torch.nn.Conv3d, z_rules.Gamma(CONFIG["conv_gamma"])),
        (torch.nn.Linear, z_rules.Gamma(CONFIG["lin_gamma"])),
    ])
    zennit_comp.register(model.visual)

    try:
        out = model(
            input_ids=output_ids,
            pixel_values=pixel_values,
            image_grid_thw=tmp_inputs.image_grid_thw,
            attention_mask=torch.ones_like(output_ids),
        )
        target_logits = out.logits[0, logit_positions, target_token_ids]
        target_logits.backward(torch.ones_like(target_logits))

        if pixel_values.grad is None or pixel_values.grad.isnan().any():
            return None

        heatmap = compute_relevance_map(
            pixel_values.grad, pixel_values, grid_t, grid_h, grid_w
        )
        return {"heatmap": heatmap, "image": image}

    except Exception as e:
        print(f"  LRP error [{sample_id}]: {e}")
        return None
    finally:
        zennit_comp.remove()
        model.zero_grad(set_to_none=True)

In [6]:
# ── Load dataset to reconstruct image paths ───────────────────────────────────
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

dataset_path = Path(CONFIG["dataset_path"])
ann_path = dataset_path / "annotations" / "test.jsonl"
dataset = load_jsonl(ann_path)
id_to_image = {item["id"]: str(IMG_DIR / item["image"]) for item in dataset}
print(f"Image paths indexed for {len(id_to_image)} samples")

Image paths indexed for 200 samples


In [7]:
def run_lrp_visual(mode):
    mode_key = mode.replace("+", "_")
    cache_dir = CACHE_DIR / mode_key
    out_dir = OUT_DIR / f"visual_{mode_key}"
    overlay_dir = out_dir / "overlay"
    out_dir.mkdir(parents=True, exist_ok=True)
    overlay_dir.mkdir(parents=True, exist_ok=True)

    manifest = json.load(open(cache_dir / "manifest.json"))
    failures = []

    for entry in tqdm(manifest, desc=f"LRP visual [{mode}]"):
        sample_id = entry["id"]
        npy_file = out_dir / f"{sample_id}.npy"
        meta_file = out_dir / f"{sample_id}.json"
        if npy_file.exists():
            continue

        try:
            record = json.load(open(entry["path"]))
            record["_image_path"] = id_to_image.get(sample_id)
            result = compute_visual_lrp(record, mode)

            if result is None:
                failures.append({"id": sample_id, "error": "lrp returned None"})
                continue

            heatmap = result["heatmap"]
            image = result["image"]

            np.save(npy_file, heatmap.astype(np.float32))
            overlay_path = overlay_dir / f"{sample_id}.png"
            save_overlay(image, heatmap, overlay_path)

            meta = {
                "id": sample_id,
                "mode": mode,
                "heatmap_path": str(npy_file),
                "overlay_path": str(overlay_path),
                "shape": list(heatmap.shape),
            }
            with open(meta_file, "w") as f:
                json.dump(meta, f, indent=2)

        except Exception as e:
            print(f"  FAIL {sample_id}: {e}")
            traceback.print_exc()
            failures.append({"id": sample_id, "error": str(e)})
        finally:
            torch.cuda.empty_cache()

    with open(out_dir / "failures.json", "w") as f:
        json.dump(failures, f, indent=2)
    n_ok = len(list(out_dir.glob("*.npy")))
    print(f"Mode '{mode}': {n_ok} heatmaps saved, {len(failures)} failures")

In [8]:
for mode in CONFIG["modes"]:
    run_lrp_visual(mode)
print("\nVisual LRP done.")

LRP visual [image]: 100%|██████████| 158/158 [00:00<00:00, 3430.85it/s]


Mode 'image': 0 heatmaps saved, 158 failures


LRP visual [image+text]: 100%|██████████| 158/158 [00:00<00:00, 2651.89it/s]

Mode 'image+text': 0 heatmaps saved, 158 failures

Visual LRP done.
